# Per-video GCaMP analysis pipeline

This notebook analyzes every Suite2p video below `EXPERIMENT_ROOT` independently. It writes each video's metrics, figures, and a versioned `*_analysis_summary.json` artifact for later comparison. It does not configure groups, aggregate folders, register longitudinal recordings, or compare treatments.

In [ ]:
from pathlib import Path
import sys

WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR if (WORKING_DIR / 'gcamp_analysis').is_dir() else WORKING_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_config, load_model
from gcamp_analysis.experiments.tree import ExperimentTreeBuilder, is_video_dir, print_tree
from gcamp_analysis.video_runner import VideoPipelineRunner
from gcamp_analysis.experiments.processor import ExperimentProcessor
from gcamp_analysis.experiments.artifacts import discover_summary_artifacts

In [ ]:
config_path = PROJECT_ROOT / 'config' / 'notebook_config.yaml'
config = load_config(config_path)
print(f'Config: {config_path}')

In [ ]:
roi_model, roi_cfg = load_model(config['models'], which='roi')
spike_model, spike_cfg = load_model(config['models'], which='spike')
models = {
    'roi': roi_model,
    'roi_config': roi_cfg,
    'spike': spike_model,
    'spike_config': spike_cfg,
}
runner = VideoPipelineRunner.build(config, models)
print(f'ROI model:   {type(roi_model).__name__}')
print(f'Spike model: {type(spike_model).__name__}')

In [ ]:
EXPERIMENT_ROOT = Path(r'C:\\path\\to\\videos')  # change for each analysis batch
DRY_RUN = True  # False writes per-video outputs
assert EXPERIMENT_ROOT.exists(), f'Experiment root not found: {EXPERIMENT_ROOT}'

tree = ExperimentTreeBuilder(is_video_dir=is_video_dir).build(EXPERIMENT_ROOT)
print_tree(tree)

In [ ]:
processor = ExperimentProcessor(
    runner=runner,
    output_root=EXPERIMENT_ROOT,
    dry_run=DRY_RUN,
    analysis_metadata={
        'config': config,
        'sensor_type': config.get('traces', {}).get('sensor_type'),
    },
)
processor.process_videos(tree, verbose=True)

In [ ]:
processed = [node for node in tree.iter_nodes() if node.payload is not None]
print(f'Analyzed {len(processed)} video(s).')
if DRY_RUN:
    print('Dry run complete: no files were written.')
else:
    summaries = discover_summary_artifacts(EXPERIMENT_ROOT)
    print(f'Wrote {len(summaries)} comparison-ready video summary artifact(s).')